# m02_l03_json

Exobean Engineering. Run the cells from top to bottom.

In [ ]:
# Google Colab only: fetches the free code companion, then run the rest.
import io, os, sys, urllib.request, zipfile
URL = "https://github.com/exobean/ai-agents-level1/releases/latest/download/shopassist-companion.zip"
if "google.colab" in sys.modules and not os.path.isdir("shopassist"):
    try:
        data = io.BytesIO(urllib.request.urlopen(URL).read())
    except OSError:  # network blocks the download: upload the zip
        from google.colab import files
        data = next(iter(files.upload()))
    zipfile.ZipFile(data).extractall(".")
    os.chdir(next(d for d in os.listdir(".") if d.endswith("-companion")))
    sys.path.insert(0, os.getcwd())


## Json

In [ ]:
import json

from shopassist.client import post_json
from shopassist.mockserver import MockProvider
from shopassist.structured import check, extract_json

body = {"model": "mock", "max_tokens": 200,
        "system": "Read the customer message. Reply with JSON only.",
        "messages": [{"role": "user", "content": "Where's #1042?!"}]}

with MockProvider() as server:
    text = post_json(server.url, body, "test-key")["content"][0]["text"]
print("raw reply:", repr(text[:38]), "...")

try:
    json.loads(text)                      # the naive way
except json.JSONDecodeError as err:
    print("json.loads failed:", err.msg)

data = extract_json(text)                 # the careful way
print("extracted:", data)
print("problems: ", check(data, {"order_id": str, "intent": str}))
